In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
df = pd.read_csv("insurance_data.csv")
df.head()

,age,affordability,bought_insurance
0,22,1,0
1,25,0,0
2,47,1,1
3,52,1,1
4,46,1,1


In [3]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df[['age','affordability']],df.bought_insurance,test_size=0.2,random_state=25)

In [4]:
len(X_train)

20

In [5]:
X_train_scaled= X_train.copy()
X_train_scaled['age']= X_train_scaled['age']/100

X_test_scaled= X_test.copy()
X_test_scaled['age']= X_test_scaled['age']/100



In [6]:
X_train_scaled

,age,affordability
0,0.22,1
13,0.49,1
6,0.60,1
17,0.19,1
23,0.50,1
19,0.21,1
24,0.54,0
16,0.58,1
20,0.26,0
3,0.52,1


In [7]:

model = keras.Sequential([
    keras.layers.Dense(
        1,
        input_shape=(2,),       
        activation='sigmoid',
        kernel_initializer='ones',
        bias_initializer='zeros'
    )
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train_scaled,y_train,epochs=500)


Epoch 1/500
1/1 [==============================] - 0s 238ms/step - loss: 0.7634 - accuracy: 0.4000
Epoch 2/500
1/1 [==============================] - 0s 3ms/step - loss: 0.7629 - accuracy: 0.4000
Epoch 3/500
1/1 [==============================] - 0s 4ms/step - loss: 0.7624 - accuracy: 0.4000
Epoch 4/500
1/1 [==============================] - 0s 4ms/step - loss: 0.7618 - accuracy: 0.4000
Epoch 5/500
1/1 [==============================] - 0s 3ms/step - loss: 0.7613 - accuracy: 0.4000
Epoch 6/500
1/1 [==============================] - 0s 3ms/step - loss: 0.7608 - accuracy: 0.4000
Epoch 7/500
1/1 [==============================] - 0s 4ms/step - loss: 0.7603 - accuracy: 0.4000
Epoch 8/500
1/1 [==============================] - 0s 3ms/step - loss: 0.7598 - accuracy: 0.4000
Epoch 9/500
1/1 [==============================] - 0s 4ms/step - loss: 0.7593 - accuracy: 0.4000
Epoch 10/500
1/1 [==============================] - 0s 3ms/step - loss: 0.7588 - accuracy: 0.4000
Epoch 11/500
1/1 [=========

In [8]:
model.evaluate(X_test_scaled,y_test)

1/1 [==============================] - 0s 168ms/step - loss: 0.5623 - accuracy: 0.8333


[0.562300443649292, 0.8333333134651184]

In [9]:
model.predict(X_test_scaled) #more than 0.5 means he will buy the insurance

1/1 [==============================] - 0s 57ms/step


array([[0.65371585],
       [0.62825775],
       [0.64442956],
       [0.43246832],
       [0.6641867 ],
       [0.41965082]], dtype=float32)

In [10]:
X_test_scaled


,age,affordability
2,0.47,1
10,0.28,1
21,0.40,1
11,0.27,0
14,0.55,1
9,0.18,0


In [11]:
coef,intercept = model.get_weights() #using tensorflow function!!!
coef,intercept

(array([[0.58243793],
        [0.7907129 ]], dtype=float32),
 array([-0.42904574], dtype=float32))

In [12]:
import math
def sigmoid(x):
    return 1/(1+math.exp(-x))


In [13]:
def prediction_function(age,affordability):
    weighted_sum= coef[0]*age+ coef[1]*affordability +intercept
    return sigmoid(weighted_sum)

In [14]:
prediction_function(0.47,1)

C:\Users\Dell\AppData\Local\Temp\ipykernel_38044\4182413728.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return 1/(1+math.exp(-x))


0.6537158262135281

In [15]:

def sigmoid_numpy(X):
   return 1/(1+np.exp(-X))

sigmoid_numpy(np.array([12,0,1]))

array([0.99999386, 0.5       , 0.73105858])

In [16]:
def log_loss(y_true, y_predicted):
    epsilon = 1e-15
    y_predicted_new = [max(i,epsilon) for i in y_predicted]
    y_predicted_new = [min(i,1-epsilon) for i in y_predicted_new]
    y_predicted_new = np.array(y_predicted_new)
    return -np.mean(y_true*np.log(y_predicted_new)+(1-y_true)*np.log(1-y_predicted_new))

In [39]:
class myNN:
    def __init__(self):
        self.w1 = 1 
        self.w2 = 1
        self.bias = 0
        
    def fit(self, X, y, epochs, loss_threshold):
        self.w1, self.w2, self.bias = self.gradient_descent(X['age'],X['affordability'],y, epochs, loss_threshold)
        print(f"Final weights and bias: w1: {self.w1}, w2: {self.w2}, bias: {self.bias}")
        
    def predict(self, X_test):
        weighted_sum = self.w1*X_test['age'] + self.w2*X_test['affordability'] + self.bias
        return sigmoid_numpy(weighted_sum)

    def gradient_descent(self, age,affordability, y_true, epochs, loss_threshold):
        w1 = w2 = 1
        bias = 0
        rate = 0.5
        n = len(age)
        for i in range(epochs):
            weighted_sum = w1 * age + w2 * affordability + bias
            y_predicted = sigmoid_numpy(weighted_sum)
            loss = log_loss(y_true, y_predicted)
            
            w1d = (1/n)*np.dot(np.transpose(age),(y_predicted-y_true)) 
            w2d = (1/n)*np.dot(np.transpose(affordability),(y_predicted-y_true)) 

            bias_d = np.mean(y_predicted-y_true)
            w1 = w1 - rate * w1d
            w2 = w2 - rate * w2d
            bias = bias - rate * bias_d
            
            if i%50==0:
                print (f'Epoch:{i}, w1:{w1}, w2:{w2}, bias:{bias}, loss:{loss}')
            
            if loss<=loss_threshold:
                print (f'Epoch:{i}, w1:{w1}, w2:{w2}, bias:{bias}, loss:{loss}')
                break

        return w1, w2, bias

In [41]:
customModel= myNN()
customModel.fit(X_train_scaled, y_train, epochs=1000, loss_threshold=0.3831)

Epoch:0, w1:0.9417429336237633, w2:0.9598996358150476, bias:-0.16004975281869202, loss:0.7633872499715488
Epoch:50, w1:0.6060686808123442, w2:2.020345448505245, bias:-1.8348868363445705, loss:0.45240151349266433
Epoch:100, w1:0.7190365469628195, w2:2.7178083052742927, bias:-2.459828235670774, loss:0.4154329666708284
Epoch:150, w1:0.900735893606065, w2:3.143604374789744, bias:-2.9063994311359393, loss:0.39857814157661764
Epoch:200, w1:1.1086275928433023, w2:3.4429260038697063, bias:-3.259953424820108, loss:0.3881610059598234
Epoch:233, w1:1.251302229923289, w2:3.5998230495873687, bias:-3.4607452771225478, loss:0.38296420591007296
Final weights and bias: w1: 1.251302229923289, w2: 3.5998230495873687, bias: -3.4607452771225478


In [42]:

coef, intercept


(array([[0.58243793],
        [0.7907129 ]], dtype=float32),
 array([-0.42904574], dtype=float32))

In [43]:

X_test_scaled

,age,affordability
2,0.47,1
10,0.28,1
21,0.40,1
11,0.27,0
14,0.55,1
9,0.18,0


In [44]:
customModel.predict(X_test_scaled)


2     0.674188
10    0.619975
21    0.654663
11    0.042173
14    0.695782
9     0.037851
dtype: float64

In [45]:
model.predict(X_test_scaled) //tensorflow 


1/1 [==============================] - 0s 20ms/step


array([[0.65371585],
       [0.62825775],
       [0.64442956],
       [0.43246832],
       [0.6641867 ],
       [0.41965082]], dtype=float32)

In [ ]:
Above you can compare predictions from our own #custom model and tensoflow model. You will notice that predictions are almost same